In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install datasets

In [3]:
from transformers import BertForTokenClassification, BertTokenizer, Trainer, TrainingArguments, BertTokenizerFast
from datasets import load_dataset, Dataset,  DatasetDict
import torch
from torch.utils.data import DataLoader
import pandas as pd
import os
from collections import Counter
import numpy as np
import string
import torch.nn as nn

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Read Data

In [5]:
os.chdir('/content/drive/My Drive/NLP-Letters-V2/notebooks/')
df1 = pd.read_csv('../data/letters_2021_processed_with_gender.csv')

In [6]:
df1['full_text'].iloc[8]

'i am pleased to have the honor of supporting the application of identifier for your residency program. i have been a mentor to mr. identifier for the past year. i am an anesthesiologist at mercy clinic in springfield, missouri. as a medical director of our large group, i have been directly involved in education and recruitment for several years. i have also served on admission committees as a medical student and during my time as chief resident. i am actively involved with educating medical students, srna’s, and anesthesiologist assistant students on rotation at our institution. mr. identifier is identifier of the most qualified and capable candidates for a career in anesthesiology that i have ever met. [ first came to know mr. identifier during his month-long rotation as a third-year medical student. from day one of his rotation he exhibited a genuine interest and motivation for learning about the science and art of anesthesiology. he consistently showed up early and was prepared for

In [7]:
df2 = pd.read_csv('../data/sentence_sets_trimmed_processed_with_gender.csv', encoding='mac-roman')

In [8]:
df2['full_text'].iloc[28]

'dear program director : i am writing to support the application of first_name last_name , a fourth year medical student in the college of medicine at suny downstate . department letter as chairman of the department of medicine , i meet individually with all applicants to internal medicine to discuss their future training and i interact with them in several small group teaching venues . in my individual meetings , we typically discuss the student\'s reasons for choosing his / his career path , long - term interests and plans , experience in research , community service , or leadership positions , concerns about his / his academic record , and his / his residency application process and strategy . i write the departmental letter in accordance with the cdim - apdim guidelines . students request the letter and unless noted otherwise , waive their right to review it . the letter is based on our personal knowledge of the student , grades in the core internal medicine clerkship , nbme subjec

In [9]:
df1['s2'].iloc[63]

"she stood out to me for her bright and possible_identifier personality, can-do attitude, and drive. i was impressed by her attention to detail, and her ability to navigate the system as a medical student to help facilitate patient care. her fund of knowledge in the basic and clinical sciences was excellent, and she often made insightful contributions to group discussions. her written and oral presentations were very well organized, thorough, and succinct, and her clinical reasoning was well-considered, well-researched, and very strong. she is currently a pgy1 doing a transitional intern year at bronx care hospital (formerly bronx lebanon). as she mentions in her personal statement, she realized in her 4th year of medical school how much she loved anesthesiology during an elective rotation in anesthesiology. she enjoyed the incorporation of math, physiology, and pharmacology with each unique case, and recalled the pivotal roles that anesthesiologists had in her life where they were the

# Load Pretrained Model

In [96]:
# Load a pretrained BERT model for token classification
model_name = "bert-base-uncased"  # You can use any model from Hugging Face
model = BertForTokenClassification.from_pretrained(model_name, num_labels=2)

# Load the BERT tokenizer
tokenizer = BertTokenizer.from_pretrained(model_name)

Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [97]:
for param in model.bert.parameters():
    param.requires_grad = False  # Freeze BERT weights


In [98]:
for name, param in model.bert.named_parameters():
    if "encoder.layer.10" in name or "encoder.layer.11" in name:
        param.requires_grad = True

# Generate Dataset

In [99]:
sentences = pd.read_csv("../data/token_clf_sentences.csv")
sentences = sentences['sentence'].tolist()

In [134]:
new_sentences = [
    "masculine", "masculinely", "masculinity", "masculism", "masculinist", "patriarch", "patriarchal",
    "male", "males", "maleness", "boy", "boys", "gamine", "boyhood", "boyish", "boyishly", "boyishness",
    "man", "men", "manhood", "manliness", "manly", "virile", "virility", "sir", "sirs", "widower", "widowers",
    "mr", "mister", "gentleman", "gentlemen", "gentlemanlike", "lord", "lords", "groom", "grooms", "bridegroom",
    "bridegrooms", "best man", "best men", "fiance", "fiancé", "fiances", "fiancés", "housewifely", "bloke", "blokes",
    "dude", "dudes", "fella", "fellas", "chap", "chaps", "guy", "guys", "lad", "lads", "macho", "machos", "actor", "actors",
    "waiter", "waiters", "king", "kings", "prince", "princes", "monk", "monks", "brethren", "monastery", "friary", "count",
    "counts", "wizard", "wizards", "priest", "priests", "prophet", "prophets", "patron", "patrons", "host", "hosts",
    "viscount", "viscounts", "shepherd", "shepherds", "steward", "stewards", "heir", "heirs", "baron", "barons", "abbot",
    "abbots", "emperor", "emperors", "traitor", "traitors", "duke", "dukes", "enchanter", "enchanters", "songster", "songsters",
    "hero", "heroes", "sultan", "sultans", "czar", "czars", "signor", "signors", "benefactor", "benefactors", "hunter",
    "tempter", "tempters", "master", "masters", "manservant", "manservants", "landlord", "landlords", "countryman", "countrymen",
    "milkman", "milkmen", "giant", "giants", "mayor", "mayors", "conductor", "god", "gods", "merman", "mermen", "oarsman", "oarsmen",
    "maid", "maids", "bellboy", "bellboys", "dairyman", "dairymen", "schoolmaster", "schoolmasters", "headmaster", "proprietor",
    "proprietors", "ambassador", "ambassadors", "adventurer", "adventurers", "protector", "protectors", "seducer", "seducers",
    "sculptor", "sculptors", "congressman", "congressmen", "sorcerer", "sorcerers", "launderer", "launderers", "anchorite",
    "anchorites", "procurer", "procurers", "elector", "electors", "adulterer", "adulterers", "doorman", "doormen", "policeman",
    "policemen", "fireman", "firemen", "businessman", "businessmen", "barman", "barmen", "mailman", "mailmen", "postman",
    "postmen", "salesman", "salesmen", "cowboy", "cowboys", "schoolboy", "schoolboys", "chairman", "chairmen", "englishman",
    "englishmen", "spokesman", "spokesmen", "councilman", "councilmen", "choirboy", "choirboys", "playboy", "playboys", "homeboy",
    "homeboys", "madman", "madmen", "anchorman", "anchormen", "sportsman", "sportsmen", "cameraman", "cameramen", "fisherman",
    "fishermen", "serviceman", "servicemen", "churchman", "churchmen", "clergyman", "clergymen", "craftsman", "craftsmen",
    "superman", "supermen", "nobleman", "noblemen", "horseman", "horsemen", "stuntman", "stuntmen", "townsman", "townsmen",
    "statesman", "statesmen", "foreman", "foremen", "kinsman", "kinsmen", "bondman", "bondmen", "bondsman", "bondsmen",
    "handyman", "handymen", "henchman", "henchmen", "strongman", "strongmen", "charman", "charmen", "tailor", "tailors",
    "newsman", "newsmen", "father", "fathers", "brother", "brothers", "husband", "husbands", "housewife", "housewives", "hubby",
    "hubbies", "son", "sons", "dad", "dads", "papa", "papas", "daddy", "daddies", "pa", "paternity", "paternally", "paternal",
    "fatherhood", "brotherhood", "fraternity", "fraternities", "fraternal", "fraternally", "brotherly", "fatherly", "grandfatherly",
    "fathered", "stepfather", "stepfathers", "stepbrother", "stepbrothers", "stepson", "stepsons", "grandfather", "grandfathers",
    "grandpa", "grandpas", "grandson", "grandsons", "nephew", "nephews", "brother-in-law", "brothers-in-law", "father-in-law",
    "fathers-in-law", "son-in-law", "sons-in-law", "uncle", "uncles", "boyfriend", "boyfriends", "ancestor", "ancestors",
    "bachelor", "bachelors", "he", "him", "his", "himself", "gay", "gays", "homosexuality", "damsel", "damsels", "maiden", "maidens"
]


In [135]:
sentences.extend(new_sentences)

In [136]:
data = {
  'tokens': [],
  'labels': []
}

# Define gendered keywords (lowercase only!)
def clean_token(token):
    return token.strip(string.punctuation).lower()  # removes .,?! etc.


def add_sentence(sentence, data):
    # Tokenize the sentence (simple split)

    tokens = sentence.lower().split()
    labels = [1 if clean_token(token) in gendered_words else 0 for token in tokens]

    # Add to dataset
    data['tokens'].append(tokens)
    data['labels'].append(labels)

    # # Optional printout for confirmation
    # print("Added:", list(zip(tokens, labels)))

In [137]:
for sentence in sentences:
  add_sentence(sentence, data)

In [138]:
# Convert to Hugging Face Dataset format
dataset = Dataset.from_dict(data)

# # Tokenizer initialization
tokenizer = BertTokenizerFast.from_pretrained(model_name)

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        padding='max_length',
        is_split_into_words=True
    )

    labels = []
    for i, label in enumerate(examples["labels"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            else:
                label_ids.append(label[word_idx])  # label all subwords
        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs



# Apply the function to the dataset
tokenized_datasets = dataset.map(tokenize_and_align_labels, batched=True)

# Display tokenized dataset
print(tokenized_datasets[0])


Map:   0%|          | 0/1697 [00:00<?, ? examples/s]

{'tokens': ['he', 'was', 'the', 'most', 'reliable', 'member', 'of', 'the', 'research', 'team.'], 'labels': [-100, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -

In [139]:
# Split dataset into train and test (80% train, 20% test)
train_test_split = tokenized_datasets.train_test_split(test_size=0.2, seed=42)

# Create a DatasetDict to store the splits
tokenized_datasets = DatasetDict({
    "train": train_test_split["train"],
    "validation": train_test_split["test"]  # Use 'validation' instead of 'test' for consistency
})

# Display dataset sizes
print(f"Train size: {len(tokenized_datasets['train'])}")
print(f"Validation size: {len(tokenized_datasets['validation'])}")

Train size: 1357
Validation size: 340


In [140]:
class WeightedLossTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")

        # Forward pass
        outputs = model(**inputs)
        logits = outputs.get("logits")  # shape: (batch, seq_len, num_labels)

        # Compute class weights from dataset
        label_counts = Counter()
        for example in tokenized_datasets["train"]:
            label_counts.update(example["labels"])

        total = sum(label_counts.values())
        weights = [1 - (label_counts.get(i, 0) / total) for i in range(model.config.num_labels)]
        weight = torch.tensor(weights).to(logits.device)

        # Define loss function (ignore_index=-100 by default)
        loss_fct = nn.CrossEntropyLoss(weight=weight, ignore_index=-100)

        # Flatten logits and labels
        logits = logits.view(-1, model.config.num_labels)     # (batch * seq_len, num_labels)
        labels = labels.view(-1)                              # (batch * seq_len)

        # ✅ Just pass it directly – CrossEntropyLoss will ignore -100 labels safely
        loss = loss_fct(logits, labels)

        return (loss, outputs) if return_outputs else loss


In [141]:
# Define training arguments with evaluation
training_args = TrainingArguments(
    output_dir='./results',           # Output directory
    num_train_epochs=3,               # Number of training epochs
    per_device_train_batch_size=2,    # Batch size for training
    per_device_eval_batch_size=2,     # Batch size for evaluation
    logging_dir='./logs',             # Directory for storing logs
    logging_steps=10,                 # Log every 10 steps
    evaluation_strategy="epoch",      # Evaluate once per epoch
    save_strategy="epoch",            # Save model at each epoch
    save_total_limit=1,               # Keep only the best model (older ones are deleted)
    load_best_model_at_end=True,      # Load the best model at the end of training
    metric_for_best_model="eval_loss",# Choose metric to track best model
    greater_is_better=False,          # Lower eval_loss is better
    report_to="none"
)

# Define the Trainer with an evaluation dataset
trainer = WeightedLossTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],  # Training set
    eval_dataset=tokenized_datasets["validation"],  # Validation set
)

# Fine-tune the model
trainer.train()

/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1594: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Epoch,Training Loss,Validation Loss
1,0.068500,0.071719
2,0.012400,0.059577
3,0.056900,0.067864


TrainOutput(global_step=2037, training_loss=0.04568803208609462, metrics={'train_runtime': 3977.4457, 'train_samples_per_second': 1.024, 'train_steps_per_second': 0.512, 'total_flos': 1063739096672256.0, 'train_loss': 0.04568803208609462, 'epoch': 3.0})

In [149]:
def predict_gendered(tokens):
    encoding = tokenizer(tokens, is_split_into_words=True, return_offsets_mapping=True, truncation=True)
    word_ids = encoding.word_ids()

    inputs = {k: torch.tensor([v]).to(device) for k, v in encoding.items() if k in tokenizer.model_input_names}

    with torch.no_grad():
        outputs = model(**inputs)
    predictions = torch.argmax(outputs.logits, dim=-1)[0]

    token_ids = inputs["input_ids"][0]
    results = []
    for idx in range(len(token_ids)):
        token = tokenizer.convert_ids_to_tokens(token_ids[idx].item())
        label = predictions[idx].item()
        results.append((token, label))

    return results


text = "He is also a great actor and father and doctor"
tokens = text.lower().split()  # match training format
tokens = [clean_token(tok) for tok in text.split()]

prediction = predict_gendered(tokens)
print("Predictions:", prediction)

Predictions: [('[CLS]', 0), ('he', 1), ('is', 0), ('also', 0), ('a', 0), ('great', 0), ('actor', 1), ('and', 0), ('father', 1), ('and', 0), ('doctor', 1), ('[SEP]', 0)]
